# N119 · 滚动哈希与子串比较

**目标：** 理解哈希相等不是字符串相等的严格证明。

**先修：** N031, N042, N128。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 多项式前缀哈希；幂表；模数；碰撞；双哈希；精确复核代价。

## 从问题到算法

多项式前缀哈希把子串比较先压为整数比较：H(l,r)=P[r]−P[l]·B^(r−l)。但哈希相等不是字符串相等，必须说明碰撞风险。本章采用“哈希分桶＋原串精确比较”，因此即使故意制造碰撞也不误判，只会变慢。重复子串长度具有单调可行性，可二分答案。

## 最小实现

**本章接口：** `RollingHash`、`find_duplicate_length(s, length)`、`longest_duplicate_substring(s)`

In [1]:
class RollingHash:
    def __init__(self,s,base=911382323,mod=1000000007):
        if mod<=0: raise ValueError('positive modulus required')
        self.s=s; self.base=base; self.mod=mod; self.prefix=[0]; self.power=[1]
        for c in s:
            self.prefix.append((self.prefix[-1]*base+ord(c)+1)%mod)
            self.power.append(self.power[-1]*base%mod)
    def query(self,left,right):
        if not 0<=left<=right<=len(self.s): raise ValueError('invalid half-open range')
        return (self.prefix[right]-self.prefix[left]*self.power[right-left])%self.mod

def find_duplicate_length(s,length,*,base=911382323,mod=1000000007):
    if length<0: raise ValueError('nonnegative length required')
    if length==0: return ''
    if length>len(s): return None
    rh=RollingHash(s,base,mod); buckets={}
    for i in range(len(s)-length+1):
        key=rh.query(i,i+length)
        for j in buckets.get(key,[]):
            if s[i:i+length]==s[j:j+length]: return s[i:i+length]
        buckets.setdefault(key,[]).append(i)
    return None

def longest_duplicate_substring(s):
    low,high=0,len(s); best=''
    while low<=high:
        mid=(low+high)//2; found=find_duplicate_length(s,mid)
        if found is None: high=mid-1
        else: best=found; low=mid+1
    return best

## 正确性、前提与复杂度

前缀相减抵消区间外字符，乘幂对齐指数。哈希仅是必要条件，原串比较提供充分条件，所以碰撞不会影响正确性。长度L存在重复，则取其前L−1字符也重复，因此二分判定满足单调性；重复允许重叠。

**代价：** 预处理O(n)，单次哈希查询O(1)固定字长模运算。碰撞桶内精确比较会增加字符代价；常见无恶意碰撞时二分接近O(n log n)，但不保证该最坏界。保守最坏O(n³ log n)，辅助O(n)。固定双哈希也不能替代一般性精确保证。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

s='banana'; rh=RollingHash(s)
show_table(['起点','三字符子串','哈希'],[(i,s[i:i+3],rh.query(i,i+3)) for i in range(len(s)-2)])
print('最长重复',longest_duplicate_substring(s))

起点,三字符子串,哈希
0,ban,708134842
1,ana,693552746
2,nan,58761225
3,ana,693552746


最长重复 ana


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert longest_duplicate_substring('banana')=='ana'
assert longest_duplicate_substring('aaaa')=='aaa'
# 模1让所有哈希值碰撞，但精确版仍不误判。
assert find_duplicate_length('abcd',2,base=1,mod=1) is None
assert find_duplicate_length('abca',1,base=1,mod=1)=='a'
from itertools import product
for n in range(7):
    for chars in product('ab',repeat=n):
        s=''.join(chars); best=0
        for length in range(n+1):
            parts=[s[i:i+length] for i in range(n-length+1)]
            exists=length==0 or len(set(parts))<len(parts)
            found=find_duplicate_length(s,length,base=1,mod=2)
            assert (found is not None)==exists
            if exists: best=length
        answer=longest_duplicate_substring(s)
        assert len(answer)==best
        if answer: assert sum(s.startswith(answer,i) for i in range(n))>=2
print('N119: 所有本章断言通过')

N119: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 用小模数主动制造碰撞。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 比较双哈希概率控制和逐字复核。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 1044. Longest Duplicate Substring（canonical）
- 187. Repeated DNA Sequences（canonical）
- 1316. Distinct Echo Substrings（extension）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。